In [ ]:
import sys
from pathlib import Path
# notebooks/ is one level below the project root; src/ is at the same level
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))


# Comparative Time Series Analysis and Forecasting of Mobile Network Traffic

**Course**: Machine Learning Techniques I  
**Assignment**: Formative 1  

---

## Introduction

### Problem Overview

The exponential growth of mobile Internet usage has made accurate forecasting of network traffic a critical challenge for telecommunications operators. The ability to predict future traffic demand enables operators to dynamically allocate radio resources, plan network capacity, and reduce congestion — directly impacting user Quality of Experience (QoE). This assignment addresses the problem of **spatiotemporal mobile network traffic forecasting** using a real-world dataset collected over the city of Milan, Italy.

Mobile network traffic exhibits complex temporal dynamics: strong daily periodicity driven by human activity rhythms, weekly cycles reflecting work-versus-leisure patterns, non-stationary trends, and irregular spikes caused by large gatherings, public holidays, or special events. These characteristics make mobile traffic a challenging but well-structured target for time series analysis and sequential machine learning methods.

### Objectives

This assignment is structured around three inter-dependent tasks:

1. **Task 1 — Data Handling & Memory Management**: The raw dataset is approximately 5 GB in size across 62 daily files. The first objective is to demonstrate efficient data engineering: loading, filtering, aggregating, and persisting the data in a way that is tractable on consumer hardware without sacrificing analytical completeness.

2. **Task 2 — Exploratory Data Analysis (EDA)**: Before any modelling, a thorough characterisation of the data is performed. This includes statistical analysis of the traffic distribution, time series visualisation, stationarity testing, decomposition into trend and seasonal components, autocorrelation analysis, spatial heatmapping, and anomaly detection. The insights derived from EDA directly inform the design choices made in Task 3.

3. **Task 3 — Forecasting Models**: Three one-step-ahead forecasting algorithms are designed, trained, and evaluated: one classical statistical model (SARIMA) and two neural network-based models (LSTM and CNN-LSTM). Each model is applied independently to three geographical areas of interest, and their predictive performance is rigorously compared using MAE, MAPE, and RMSE over a held-out test week (December 16–22, 2013).

---

### Dataset Description

The dataset used in this study was released by Telecom Italia Mobile (TIM) as part of the **Telecom Italia Big Data Challenge** and is publicly available via the Harvard Dataverse repository [1, 2]. A full description is provided by Barlacchi et al. [3].

#### Spatial Structure

The city of Milan is tessellated into a **100 × 100 regular grid** of 10,000 square cells, each approximately **235 m × 235 m** in extent. The grid is georeferenced to the WGS84 coordinate system (EPSG:4326). Each cell is identified by a unique integer `square_id` (1–10,000). The grid geometry is provided as a GeoJSON file [2].

#### Temporal Structure

Telecommunications activity is recorded in **10-minute time intervals** (144 slots per day) over a continuous two-month period from **November 1 to January 1, 2014** — a total of 62 days (8,928 time slots per area).

#### Activity Measurement

Each record in the dataset reports the number of **Call Detail Records (CDRs)** generated within a given grid cell during a given 10-minute interval. A CDR is produced every time a user initiates or terminates a mobile Internet session, or when a session exceeds 15 minutes or 5 MB of data transfer. The raw CDR count is scaled by a confidential constant $k$ defined by Telecom Italia to protect commercial sensitivity, but the spatiotemporal dynamics remain fully preserved [3].

#### File Format

The dataset is distributed as 62 tab-separated plain-text files (one per day), with no header row. Each row encodes one `(square_id, time_interval, country_code)` combination. Multiple rows may share the same `(square_id, time_interval)` pair — one row per originating country code. The relevant columns for this assignment are:

| Field | Index | Type | Description |
|-------|-------|------|-------------|
| `square_id` | 0 | int | Grid cell identifier |
| `time_interval` | 1 | int | 10-min slot start (Unix ms) |
| `country_code` | 2 | int | Origin country phone code |
| `internet` | 7 | float | Internet CDR count (target) |

> **Field ordering note**: The paper [3] contains an error in the stated column order. `country_code` is the **third** field (index 2), and all subsequent fields are shifted accordingly. This was identified by cross-referencing column values against known country codes (e.g., 39 = Italy).

#### Three Focal Areas

As required by the assignment, all time series analyses and forecasting experiments focus on three specific grid areas:

| Area | `square_id` | Selection criterion |
|------|-------------|---------------------|
| A | Determined in Task 2 | Highest total internet traffic over the two-month period |
| B | 4159 | Specified by assignment (corresponds to Bocconi University area [3]) |
| C | 4556 | Specified by assignment (corresponds to Navigli nightlife district [3]) |

---

### Notebook Structure

The analysis is organised across four notebooks:

| Notebook | Task | Contents |
|----------|------|----------|
| `00_introduction.ipynb` | — | Problem overview, objectives, dataset description |
| `01_data_handling.ipynb` | Task 1 | Memory-efficient loading, optimisation, Parquet export |
| `02_eda.ipynb` | Task 2 | Full exploratory data analysis |
| `03_forecasting.ipynb` | Task 3 | Model design, training, evaluation, comparison |
| `04_conclusion.ipynb` | — | Key findings and insights |

---

### References

[1] Telecom Italia, "Telecommunications Activity Dataset — Milan," Harvard Dataverse, doi:10.7910/DVN/EGZHFV, 2015.  
[2] Telecom Italia, "Grid Dataset — Milan," Harvard Dataverse, doi:10.7910/DVN/QJWLFU, 2015.  
[3] G. Barlacchi, M. De Nadai, R. Larcher, A. Casella, C. Chitic, G. Torrisi, F. Antonelli, A. Vespignani, A. Pentland, and B. Lepri, "A multi-source dataset of urban life in the city of Milan and the Province of Trentino," *Sci. Data*, vol. 2, no. 1, p. 150055, Sep. 2015, doi:10.1038/sdata.2015.55.

In [ ]:
# Dataset summary statistics (run after Notebook 1 has been executed)
import pandas as pd
import numpy as np
from pathlib import Path

PARQUET_PATH = Path("../processed/area_timeseries.parquet")

if not PARQUET_PATH.exists():
    print("Parquet file not found. Run 01_data_handling.ipynb first.")
else:
    df = pd.read_parquet(PARQUET_PATH)
    # parquet has columns: square_id, datetime, internet_traffic

    print("=" * 55)
    print("Dataset Summary (3 focal areas)")
    print("=" * 55)
    print(f"Total records (area × time slots) : {len(df):,}")
    print(f"Unique grid areas                  : {df['square_id'].nunique():,}")
    print(f"Unique time slots                  : {df['datetime'].nunique():,}")
    print(f"Observation period                 : {df['datetime'].min().date()} → {df['datetime'].max().date()}")
    print(f"Temporal resolution                : 10 minutes (144 slots/day)")
    print(f"Spatial coverage                   : 100 × 100 grid (~235m × 235m cells)")
    print(f"Internet traffic — min             : {df['internet_traffic'].min():.4f}")
    print(f"Internet traffic — max             : {df['internet_traffic'].max():.4f}")
    print(f"Internet traffic — mean            : {df['internet_traffic'].mean():.4f}")
    print(f"Internet traffic — std             : {df['internet_traffic'].std():.4f}")
    print(f"Missing values                     : {df['internet_traffic'].isna().sum():,} ({df['internet_traffic'].isna().mean()*100:.1f}%)")
    print("=" * 55)